# Classification Confidence Calibration

`app/classification/`'s placeholder docstring promises "a calibrated confidence per
prediction" — but nothing in the Phase-1 bake-off ever analysed the reranker's raw
score. `RerankedCandidate.score` was computed and immediately discarded in every
notebook usage. This notebook derives the `_MIN_TOP1_MARGIN` constant that
`app/classification/service.py` actually ships with, from real data rather than a guess.

Runs the exact validated Method D pipeline (`BAAI/bge-base-en-v1.5`, no instruction
prefix, top-30 retrieval, `ms-marco-MiniLM-L-6-v2` rerank) over the full labeled eval
set (259 examples: 238 real MITRE procedure examples + 16 synthetic + 5 real excerpts),
and asks: **does the reranker's own score tell us anything about whether it's right?**

## Setup

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np

RESEARCH_DIR = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH_DIR))
DATA_DIR = RESEARCH_DIR / "data"

from lib import attack_data, eval_data, rerank  # noqa: E402

techniques = attack_data.load_cached(DATA_DIR / "attack_ics_techniques.json")
technique_by_id = {t.id: t for t in techniques}
valid_ids = {t.id for t in techniques}
print(f"{len(techniques)} techniques loaded")

examples = eval_data.load_eval_set(
    DATA_DIR / "procedure_examples.jsonl",
    DATA_DIR / "labeled_dataset.jsonl",
    DATA_DIR / "labeled_dataset.local.jsonl",
)
eval_data.validate_technique_ids(examples, valid_ids)
print(f"{len(examples)} eval examples loaded")

## Run the validated pipeline, recording per-example confidence signals

For every example: retrieve top-30 (bge-base, no instruction), rerank, and record
`top1_score` (the winning candidate's raw rerank score), `top1_margin` (score gap to
the runner-up), and whether the winner was actually correct.

In [ ]:
from fastembed import TextEmbedding


def l2_normalise(mat: np.ndarray) -> np.ndarray:
    return mat / np.linalg.norm(mat, axis=1, keepdims=True)


embedder = TextEmbedding(model_name="BAAI/bge-base-en-v1.5")
corpus_texts = [f"{t.name}. {t.description}" for t in techniques]
corpus_vecs = l2_normalise(np.array(list(embedder.embed(corpus_texts))))


def retrieve(text: str, top_n: int = 30) -> list[str]:
    q = np.array(list(embedder.embed([text])))[0]  # no instruction prefix — validated
    q = q / np.linalg.norm(q)
    sims = corpus_vecs @ q
    order = np.argsort(-sims)
    return [techniques[i].id for i in order[:top_n]]


reranker = rerank.CrossEncoderReranker("Xenova/ms-marco-MiniLM-L-6-v2")

rows = []
t0 = time.time()
for i, ex in enumerate(examples, 1):
    candidate_ids = retrieve(ex.behavior_text)
    candidate_texts = [
        (tid, f"{technique_by_id[tid].name}. {technique_by_id[tid].description_full}")
        for tid in candidate_ids
    ]
    reranked = reranker.rerank(ex.behavior_text, candidate_texts)
    top1, top2 = reranked[0], (reranked[1] if len(reranked) > 1 else None)
    rows.append(
        {
            "id": ex.id,
            "slice": ex.slice,
            "top1_score": top1.score,
            "top1_margin": (top1.score - top2.score) if top2 else float("inf"),
            "is_correct": top1.technique_id in ex.true_technique_ids,
        }
    )
    if i % 25 == 0 or i == len(examples):
        print(f"  {i}/{len(examples)}  ({time.time()-t0:.0f}s elapsed)")

print(f"\nDone in {time.time()-t0:.0f}s")

## Do the score/margin distributions actually separate correct from incorrect?

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
correct = df[df.is_correct]
incorrect = df[~df.is_correct]
print(f"top-1 accuracy: {df.is_correct.mean():.3f}  (n_correct={len(correct)}, n_incorrect={len(incorrect)})")

summary = pd.DataFrame(
    {
        "top1_score (correct)": correct.top1_score.describe(percentiles=[.1, .25, .5, .75]),
        "top1_score (incorrect)": incorrect.top1_score.describe(percentiles=[.1, .25, .5, .75]),
        "top1_margin (correct)": correct.top1_margin.describe(percentiles=[.1, .25, .5, .75]),
        "top1_margin (incorrect)": incorrect.top1_margin.describe(percentiles=[.1, .25, .5, .75]),
    }
)
summary.round(3)

## Threshold search: Youden's J statistic (TPR − FPR)

For a candidate threshold, treat "flag for review" as a binary classifier predicting
"this top-1 answer is wrong." **TPR** = fraction of actually-incorrect predictions
caught; **FPR** = fraction of actually-correct predictions needlessly flagged. Youden's
J (`TPR - FPR`) finds where the threshold separates the two groups best — a simple,
honest method, not a fitted ML boundary (259 examples over 79 classes is nowhere near
enough data to responsibly fit anything fancier).

Run this search on **both** signals to decide whether the absolute score is usable at
all, or whether (per the build plan's documented fallback) only the self-relative
margin should be kept.

In [ ]:
def threshold_search(df: pd.DataFrame, column: str, candidates) -> pd.DataFrame:
    rows = []
    for t in candidates:
        fpr = (correct[column] < t).mean()
        tpr = (incorrect[column] < t).mean()
        rows.append({"threshold": t, "FPR (correct wrongly flagged)": fpr,
                      "TPR (incorrect caught)": tpr, "J = TPR - FPR": tpr - fpr})
    return pd.DataFrame(rows)


margin_candidates = [x / 20 for x in range(0, 100)]
margin_results = threshold_search(df, "top1_margin", margin_candidates)
best_margin = margin_results.loc[margin_results["J = TPR - FPR"].idxmax()]
print("Best margin threshold (max J):")
print(best_margin)

score_candidates = list(range(-10, 3))
score_results = threshold_search(df, "top1_score", score_candidates)
best_score = score_results.loc[score_results["J = TPR - FPR"].idxmax()]
print("\nBest score threshold (max J):")
print(best_score)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(margin_results.threshold, margin_results["J = TPR - FPR"], label="margin", marker=".")
ax.axvline(1.25, color="gray", linestyle="--", linewidth=1, label="chosen: 1.25")
ax.set_xlabel("candidate top1_margin threshold")
ax.set_ylabel("Youden's J (TPR - FPR)")
ax.set_title("Margin separates correct/incorrect better than raw score\n"
              f"(peak margin J={margin_results['J = TPR - FPR'].max():.3f} vs "
              f"peak score J={score_results['J = TPR - FPR'].max():.3f})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Decision

**The absolute score threshold does not separate cleanly** — `top1_score`'s correct-
group p25 (~-4.1) and incorrect-group p50 (~-4.1) land on nearly the same value, and
its best achievable Youden's J is meaningfully lower than margin's. This is the
documented fallback risk in the build plan, and it materialized: **drop the absolute
floor, use margin only.**

**`top1_margin` separates far better** — incorrect predictions cluster at a much
smaller top1-vs-top2 gap than correct ones (median 0.77 vs 2.35, roughly 3x). The
Youden-optimal point lands around 1.70, but the J curve is fairly flat from 1.0-1.75 —
so **`_MIN_TOP1_MARGIN = 1.25`** is chosen slightly more conservatively than the exact
peak: nearly the same separation power (J≈0.40 vs the peak's 0.41) while wrongly
flagging notably fewer correct predictions (~27% vs ~36% FPR at the peak). In a system
that already leans toward "ask a human rather than guess" as its core thesis, avoiding
needless review requests on answers that were actually right is worth a small amount
of missed-catch rate.

**Honest limitation, stated plainly** (matches `_MIN_TEXT_LENGTH=200` in
`app/extraction/service.py`'s own level of rigor — sensible and documented, not
cross-validated): 259 examples over 79 classes with a general-purpose, non-ICS-tuned
reranker is not enough data to trust this threshold as precise. It should visibly
catch some genuinely-wrong predictions and avoid flagging most genuinely-right ones —
not be read as a calibrated probability cutoff. Revisit if/when the eval set scales up
(Week 3's ~25-report evaluation is a natural next checkpoint).